In [0]:
# append so it will have all data 
# partition by Ingestion date
# it will not maintain history as like silver
# if jobs are run twice - new data antijoin  with old data ====handle issue
# mergeSchema is done in bronze layer ===handle issue
# read file - add metadata - Metadat Antijoin oldBronze - write(mergeSchema)

In [0]:
dfraw=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/deltacatalog/deltaschema/dailydata_raw/Sales_20200322*")
dfraw.display()

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, lit, input_file_name, col

dfmetadata = dfraw.withColumn("ingestion_Timestamp", current_timestamp()) \
    .withColumn("ingestion_date", current_date()) \
    .withColumn("source", lit("raw_layer")) \
     .withColumn("file_name", col("_metadata.file_path"))

dfmetadata.display()


In [0]:
dfraw1=spark.sql("select *  from deltacatalog.deltaschema.bronze_sales")
dfraw1.display()

In [0]:
spark.sql("select distinct file_name from deltacatalog.deltaschema.bronze_sales").display()

In [0]:
df_new = dfmetadata.join(dfraw1, on="file_name", how="left_anti")
df_new.display()

In [0]:

df_new.write.format("delta") \
    .option("mergeSchema", "true") \
    .partitionBy("ingestion_date") \
    .mode("append") \
    .saveAsTable("deltacatalog.deltaschema.bronze_sales")

In [0]:
spark.sql("select * from deltacatalog.deltaschema.bronze_sales").display()

# 